# 4 — Severity encoding and ordinal coefficients

Compare the original numeric severity score with categorical indicators in all
30 Gamma log-link GLMs. Preserve the split, original labels, peril indicators,
and `log1p(Loss)` response. Miscellaneous and negligible are the reference levels.
Categorical encoding relaxes equal spacing on the log link scale.

Inputs: `intermediate/s0_data.pkl`, `s1_labels.pkl`, and `s2_analysis.pkl`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from itertools import combinations
import statsmodels.api as sm
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
INTERMEDIATE = Path("intermediate")
prepared = pd.read_pickle(INTERMEDIATE / "s0_data.pkl")
classified = pd.read_pickle(INTERMEDIATE / "s1_labels.pkl")
numeric = pd.read_pickle(INTERMEDIATE / "s2_analysis.pkl")
claims = classified["claims"]
descriptions = classified["descriptions"]
labels = classified["labels"]
fixed_labels = classified["fixed_labels"]
fixed_manifest = classified["fixed_manifest"]
original_cube = classified["original_cube"]
fixed_cube = classified["fixed_cube"]
original_ids = classified["original_ids"]
fixed_ids = classified["fixed_ids"]
MODEL_ORDER = classified["model_order"]
LABEL_NAMES = classified["label_names"]
RUNS = classified["runs"]

PERILS, FEATURES = prepared["perils"], prepared["features"]
peril_design = prepared["peril_design"]
baseline_predictions = prepared["baseline_predictions"].prediction.to_numpy()
baseline_mae = float(prepared["baseline"].query("split == 'validation'").mae.iloc[0])
numeric_coefficients = numeric["coefficients"]
numeric_predictions = numeric["validation_predictions"]

In [3]:
LEVELS = [1, 2, 3, 4, 5]
PERIL_FEATURES = PERILS[:-1]
SEED, BOOTSTRAPS = 20261107, 5000
train_mask = claims.split.eq("train").to_numpy()
valid_mask = claims.split.eq("validation").to_numpy()
validation_claims = claims.loc[valid_mask].copy()

## Fit the categorical alternative

Categories absent from training have no estimable coefficient. Qwen run 2 has no
catastrophic labels in either split. Stop if a validation label is unsupported
by training rather than assigning an arbitrary effect to it.

Numeric coefficients and predictions are loaded from Notebook 2.

In [4]:
def gamma_fit(x, y):
    return sm.GLM(y, x, family=sm.families.Gamma(link=sm.families.links.Log())).fit()


x_base = peril_design.astype(float)
fits = {}
performance_rows, prediction_frames = [], []
for model in MODEL_ORDER:
    for run in RUNS:
        feature = labels.loc[
            labels.model.eq(model) & labels.run.eq(run),
            ["description_id", "label_value"],
        ]
        data = claims.merge(
            feature, on="description_id", how="left", sort=False, validate="many_to_one"
        )
        severity = data.label_value.astype(int)
        observed_train = set(severity[train_mask])
        unsupported = set(severity[valid_mask]) - observed_train
        if unsupported:
            raise ValueError(
                f"{model}, run {run}: no training observations for {unsupported}"
            )

        dummies = pd.get_dummies(
            pd.Categorical(severity, categories=LEVELS), prefix="severity", dtype=float
        )
        columns = [
            f"severity_{level}" for level in LEVELS[1:] if level in observed_train
        ]
        x = pd.concat([x_base, dummies[columns]], axis=1)
        fit = gamma_fit(x.loc[train_mask], data.loc[train_mask, "log_loss"])
        fits[(model, run, "categorical")] = fit
        pred = np.asarray(fit.predict(x.loc[valid_mask]))
        y = data.loc[valid_mask, "log_loss"]
        performance_rows.append(
            {
                "model": model,
                "run": run,
                "encoding": "categorical",
                "split": "validation",
                "mae": mean_absolute_error(y, pred),
                "r2": r2_score(y, pred),
            }
        )

        frame = data.loc[valid_mask, ["claim_row", "description_id", "log_loss"]].copy()
        frame["model"], frame["run"] = model, run
        saved_predictions = (
            numeric_predictions.loc[
                numeric_predictions.model.eq(model) & numeric_predictions.run.eq(run)
            ]
            .set_index("claim_row")
            .prediction
        )
        frame["numeric_prediction"] = frame.claim_row.map(saved_predictions)
        frame["categorical_prediction"] = pred
        frame["numeric_abs_error"] = (frame.log_loss - frame.numeric_prediction).abs()
        frame["categorical_abs_error"] = (
            frame.log_loss - frame.categorical_prediction
        ).abs()
        frame["abs_error_change"] = (
            frame.categorical_abs_error - frame.numeric_abs_error
        )
        prediction_frames.append(frame)

performance = pd.concat(
    [
        numeric["performance"]
        .query("split == 'validation'")
        .assign(encoding="numeric"),
        pd.DataFrame(performance_rows),
    ],
    ignore_index=True,
)
validation_predictions = pd.concat(prediction_frames, ignore_index=True)

In [5]:
validation_performance = performance.query("split == 'validation'")
summary = (
    validation_performance.groupby(["model", "encoding"])
    .agg(
        mean_mae=("mae", "mean"),
        sd_mae=("mae", "std"),
        mean_r2=("r2", "mean"),
        sd_r2=("r2", "std"),
    )
    .reindex(
        pd.MultiIndex.from_product(
            [MODEL_ORDER, ["numeric", "categorical"]], names=["model", "encoding"]
        )
    )
    .reset_index()
)
summary["mae_improvement_pct_baseline"] = (
    100 * (baseline_mae - summary.mean_mae) / baseline_mae
)

comparison = summary.pivot(
    index="model", columns="encoding", values=["mean_mae", "mean_r2"]
).reindex(MODEL_ORDER)
comparison.columns = [f"{encoding}_{metric}" for metric, encoding in comparison.columns]
comparison["delta_mae"] = comparison.categorical_mean_mae - comparison.numeric_mean_mae
comparison["delta_mae_pct_baseline"] = 100 * comparison.delta_mae / baseline_mae
comparison["delta_r2"] = comparison.categorical_mean_r2 - comparison.numeric_mean_r2
comparison["numeric_mae_rank"] = comparison.numeric_mean_mae.rank(method="min").astype(
    int
)
comparison["categorical_mae_rank"] = comparison.categorical_mean_mae.rank(
    method="min"
).astype(int)

paired_runs = validation_performance.pivot(
    index=["model", "run"], columns="encoding", values=["mae", "r2"]
)
paired_runs.columns = [
    f"{encoding}_{metric}" for metric, encoding in paired_runs.columns
]
paired_runs["delta_mae"] = paired_runs.categorical_mae - paired_runs.numeric_mae
paired_runs["delta_r2"] = paired_runs.categorical_r2 - paired_runs.numeric_r2
paired_runs = paired_runs.reindex(
    pd.MultiIndex.from_product([MODEL_ORDER, RUNS], names=["model", "run"])
)

## Validation sensitivity

Changes are categorical minus numeric. Intervals use 5,000 paired resamples of
the 996 validation descriptions, retaining all claims and both encodings.
Metrics average five runs and are not scores of ensemble predictions.
Intervals condition on fitted models and the observed label runs.

In [6]:
# Aggregate sufficient statistics by description to preserve repeated-claim clusters efficiently.
valid_ids = pd.Index(validation_claims.description_id.unique())
y_clusters = (
    validation_claims.assign(y2=lambda d: d.log_loss**2)
    .groupby("description_id")
    .agg(n=("log_loss", "size"), y=("log_loss", "sum"), y2=("y2", "sum"))
    .reindex(valid_ids)
)
cluster_mae, cluster_sse, cluster_baseline_gain = [], [], []
base_errors = pd.Series(
    np.abs(validation_claims.log_loss.to_numpy() - baseline_predictions[valid_mask]),
    index=validation_claims.claim_row,
)
for model in MODEL_ORDER:
    p = validation_predictions.loc[validation_predictions.model.eq(model)].copy()
    p["sse_gain"] = (p.log_loss - p.numeric_prediction) ** 2 - (
        p.log_loss - p.categorical_prediction
    ) ** 2
    p["baseline_gain"] = p.claim_row.map(base_errors) - p.categorical_abs_error
    # Average over runs within claims before summing across description clusters.
    per_claim = (
        p.groupby(["description_id", "claim_row"])[
            ["abs_error_change", "sse_gain", "baseline_gain"]
        ]
        .mean()
        .reset_index()
    )
    grouped = (
        per_claim.groupby("description_id")[
            ["abs_error_change", "sse_gain", "baseline_gain"]
        ]
        .sum()
        .reindex(valid_ids)
    )
    cluster_mae.append(grouped.abs_error_change.to_numpy())
    cluster_sse.append(grouped.sse_gain.to_numpy())
    cluster_baseline_gain.append(grouped.baseline_gain.to_numpy())
cluster_mae, cluster_sse, cluster_baseline_gain = map(
    np.column_stack, [cluster_mae, cluster_sse, cluster_baseline_gain]
)
boot_delta_mae = np.empty((BOOTSTRAPS, len(MODEL_ORDER)))
boot_delta_r2 = np.empty_like(boot_delta_mae)
boot_baseline_gain = np.empty_like(boot_delta_mae)
rng = np.random.default_rng(SEED)
for start in range(0, BOOTSTRAPS, 200):
    stop = min(start + 200, BOOTSTRAPS)
    # Multinomial counts are exactly equivalent to resampling description IDs with replacement.
    weights = rng.multinomial(
        len(valid_ids), np.full(len(valid_ids), 1 / len(valid_ids)), size=stop - start
    )
    n = weights @ y_clusters.n.to_numpy()
    y_sum = weights @ y_clusters.y.to_numpy()
    sst = weights @ y_clusters.y2.to_numpy() - y_sum**2 / n
    boot_delta_mae[start:stop] = (weights @ cluster_mae) / n[:, None]
    boot_delta_r2[start:stop] = (weights @ cluster_sse) / sst[:, None]
    boot_baseline_gain[start:stop] = (weights @ cluster_baseline_gain) / n[:, None]

# Sufficient-statistic point estimates must match direct metrics.
uncertainty_rows = []
for j, model in enumerate(MODEL_ORDER):
    uncertainty_rows.append(
        {
            "model": model,
            "delta_mae": comparison.loc[model, "delta_mae"],
            "delta_mae_ci_low": np.quantile(boot_delta_mae[:, j], 0.025),
            "delta_mae_ci_high": np.quantile(boot_delta_mae[:, j], 0.975),
            "delta_r2": comparison.loc[model, "delta_r2"],
            "delta_r2_ci_low": np.quantile(boot_delta_r2[:, j], 0.025),
            "delta_r2_ci_high": np.quantile(boot_delta_r2[:, j], 0.975),
            "categorical_mae_gain_vs_baseline": baseline_mae
            - comparison.loc[model, "categorical_mean_mae"],
            "baseline_gain_ci_low": np.quantile(boot_baseline_gain[:, j], 0.025),
            "baseline_gain_ci_high": np.quantile(boot_baseline_gain[:, j], 0.975),
        }
    )
uncertainty = pd.DataFrame(uncertainty_rows)

display(
    comparison[["numeric_mean_mae", "categorical_mean_mae", "delta_mae", "delta_r2"]]
    .join(uncertainty.set_index("model")[["delta_mae_ci_low", "delta_mae_ci_high"]])
    .round(6)
)

,numeric_mean_mae,categorical_mean_mae,delta_mae,delta_r2,delta_mae_ci_low,delta_mae_ci_high
model,,,,,,
GPT-5.6 Sol,0.990593,0.991467,0.000873,-0.000982,-0.002997,0.004230
Gemini 3.1 Pro,0.979437,0.979976,0.000539,-0.000486,-0.003168,0.004111
Kimi K2.6,1.003742,1.002567,-0.001175,0.001867,-0.004071,0.001386
DeepSeek V4 Pro,1.005016,1.005216,0.000200,-0.000704,-0.002300,0.002450
Qwen3.5 397B,1.014297,1.013838,-0.000460,0.000666,-0.001975,0.001004
GLM-5.2,1.000580,1.000905,0.000325,0.000355,-0.002523,0.002888


In [7]:
modal = numeric["modal_wide"]
pair_rows = []
for a, b in combinations(MODEL_ORDER, 2):
    row = {
        "model_a": a,
        "model_b": b,
        "label_disagreement_rate": (modal[a] != modal[b]).mean(),
    }
    for encoding in ["numeric", "categorical"]:
        gap = abs(
            comparison.loc[a, f"{encoding}_mean_mae"]
            - comparison.loc[b, f"{encoding}_mean_mae"]
        )
        row[f"{encoding}_mae_gap_pct_baseline"] = 100 * gap / baseline_mae
        row[f"{encoding}_within_1pct"] = bool(gap < 0.01 * baseline_mae)
    pair_rows.append(row)
pair_comparison = pd.DataFrame(pair_rows)
tolerance_rows = []
for threshold in [0.5, 1.0, 2.0]:
    for encoding in ["numeric", "categorical"]:
        selected = pair_comparison.loc[
            pair_comparison[f"{encoding}_mae_gap_pct_baseline"] < threshold
        ]
        tolerance_rows.append(
            {
                "tolerance_pct_baseline": threshold,
                "encoding": encoding,
                "pairs": len(selected),
                "min_label_disagreement": selected.label_disagreement_rate.min(),
                "median_label_disagreement": selected.label_disagreement_rate.median(),
                "max_label_disagreement": selected.label_disagreement_rate.max(),
            }
        )
tolerance_summary = pd.DataFrame(tolerance_rows)
display(tolerance_summary.round(4))

,tolerance_pct_baseline,encoding,pairs,min_label_disagreement,median_label_disagreement,max_label_disagreement
0,0.5,numeric,3,0.1994,0.2330,0.2390
1,0.5,categorical,3,0.1994,0.2330,0.2390
2,1.0,numeric,6,0.1994,0.2192,0.2471
3,1.0,categorical,5,0.1994,0.2330,0.2471
4,2.0,numeric,11,0.1994,0.2390,0.4090
5,2.0,categorical,11,0.1994,0.2390,0.4090


## Appendix: ordinal coefficients and validation ranks

Standard errors are conventional model-based GLM standard errors conditional
on each run's labels. Spearman correlation compares fitted predictions with
observed transformed loss on the 1,039 validation claims. Coefficients act on
the log link for the mean of transformed loss, not expected dollar loss.

In [8]:
# Report the numeric coefficients and standard errors saved by Notebook 2.
ordinal_rows = []
for model in MODEL_ORDER:
    for run in RUNS:
        coefficient = numeric_coefficients.loc[
            numeric_coefficients.model.eq(model)
            & numeric_coefficients.run.eq(run)
            & numeric_coefficients.term.eq("label_value")
        ].iloc[0]
        p = validation_predictions.loc[
            validation_predictions.model.eq(model) & validation_predictions.run.eq(run)
        ]
        ordinal_rows.append(
            {
                "model": model,
                "run": run,
                "gamma": float(coefficient.coefficient),
                "gamma_se": float(coefficient.standard_error),
                "validation_spearman": float(
                    spearmanr(p.numeric_prediction, p.log_loss).statistic
                ),
            }
        )
ordinal_report = pd.DataFrame(ordinal_rows)
display(ordinal_report.round(4))

,model,run,gamma,gamma_se,validation_spearman
0,GPT-5.6 Sol,1,0.0927,0.0045,0.6862
1,GPT-5.6 Sol,2,0.0849,0.0044,0.6852
2,GPT-5.6 Sol,3,0.0931,0.0045,0.6726
3,GPT-5.6 Sol,4,0.0896,0.0044,0.6811
4,GPT-5.6 Sol,5,0.0880,0.0046,0.6592
5,Gemini 3.1 Pro,1,0.0722,0.0037,0.6864
6,Gemini 3.1 Pro,2,0.0716,0.0038,0.6915
7,Gemini 3.1 Pro,3,0.0775,0.0038,0.6868
8,Gemini 3.1 Pro,4,0.0839,0.0038,0.6839
9,Gemini 3.1 Pro,5,0.0762,0.0038,0.6864


## Save results

In [9]:
categorical_coefficients = pd.DataFrame(
    [
        {
            "model": model,
            "run": run,
            "encoding": encoding,
            "term": term,
            "coefficient": float(value),
            "standard_error": float(fit.bse[term]),
        }
        for (model, run, encoding), fit in fits.items()
        for term, value in fit.params.items()
    ]
)
all_coefficients = pd.concat(
    [
        numeric_coefficients.assign(encoding="numeric"),
        categorical_coefficients,
    ],
    ignore_index=True,
)
pd.to_pickle(
    {
        "comparison": comparison,
        "paired_intervals": uncertainty,
        "predictions": validation_predictions,
        "coefficients": all_coefficients,
        "ordinal_report": ordinal_report,
        "tolerances": tolerance_summary,
    },
    INTERMEDIATE / "s4_encoding_results.pkl",
)